# 🧪 Laboratório de testes — `yfinance` para a B3

Notebook de apoio para você **entender a fonte de dados** antes de usá-la no projeto
de agentes. Aqui não tem CrewAI: é só o `yfinance` puro, para você experimentar.

## O que saber de antemão
- O `yfinance` lê dados do Yahoo Finance, **de graça e sem chave de API**.
- Ações da B3 usam o sufixo **`.SA`** (ex.: `PETR4.SA`, `VALE3.SA`).
- O índice Ibovespa é **`^BVSP`** (sem `.SA`).
- As cotações têm **atraso (~15 min)** — não são tempo real tick a tick.

## 1. Instalação

In [ ]:
# !pip install yfinance

## 2. Um ativo, primeiro contato

`yf.Ticker("PETR4.SA")` cria um objeto que representa o ativo. Ele ainda não baixou
nada — os dados vêm quando chamamos um método como `.history()`.

In [1]:
import yfinance as yf

petr = yf.Ticker("PETR4.SA")
petr

yfinance.Ticker object <PETR4.SA>

## 3. Histórico de preços com `.history()`

O método mais confiável para preços. Retorna um DataFrame do pandas com as colunas
`Open, High, Low, Close, Volume`. O índice são as datas.

- `period`: janela de tempo (`1d`, `5d`, `1mo`, `6mo`, `1y`, `max`...).
- A última linha é o pregão mais recente (durante o mercado aberto, é o dia em
  formação).

In [2]:
petr.history(period="5d")

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-10-30 00:00:00-03:00,52.450001,53.660000,52.360001,52.709999,61743200,0.0,0.0
2026-11-03 00:00:00-03:00,51.200001,51.209999,49.150002,50.709999,93496300,0.0,0.0
2026-11-04 00:00:00-03:00,50.040001,50.310001,46.639999,47.119999,94219800,0.0,0.0
2026-11-05 00:00:00-03:00,47.900002,48.040001,45.950001,45.950001,53109700,0.0,0.0
2026-11-06 00:00:00-03:00,46.000000,46.110001,44.959999,45.360001,47069700,0.0,0.0


## 4. Cotação rápida com `.fast_info`

`fast_info` é um resumo leve e rápido (último preço, máxima/mínima do dia, etc.).
As chaves disponíveis podem variar com a versão da biblioteca, então imprima para ver.

In [3]:
dict(petr.fast_info)

{'currency': 'BRL',
 'dayHigh': 46.11000061035156,
 'dayLow': 44.959999084472656,
 'exchange': 'SAO',
 'fiftyDayAverage': 50.803399887084964,
 'lastPrice': 45.36000061035156,
 'lastVolume': 47069700,
 'marketCap': 619775722912.0898,
 'open': 46.0,
 'previousClose': 46.54999923706055,
 'quoteType': 'EQUITY',
 'regularMarketPreviousClose': 45.95000076293945,
 'shares': 13663485771,
 'tenDayAverageVolume': 73133120,
 'threeMonthAverageVolume': 52290501,
 'timezone': 'America/Sao_Paulo',
 'twoHundredDayAverage': 44.39189998626709,
 'yearChange': 0.42790554155305355,
 'yearHigh': 59.790000915527344,
 'yearLow': 29.530000686645508}

## 5. Dados fundamentalistas com `.info`

`.info` traz um dicionário grande (setor, nome completo, P/L, dividend yield...).
É mais lento e **às vezes volta vazio ou dá erro** dependendo do momento — por isso
envolvemos em `try/except` e por isso o projeto principal usa `.history()` para preço.

In [4]:
try:
    info = petr.info
    for chave in ["longName", "sector", "trailingPE", "dividendYield"]:
        print(f"{chave}: {info.get(chave)}")
except Exception as e:
    print("Não foi possível obter .info:", e)

longName: Petróleo Brasileiro S.A. - Petrobras
sector: Energy
trailingPE: 4.324118
dividendYield: 7.97


## 6. Períodos e intervalos diferentes

Além do `period`, você pode escolher o `interval` (granularidade). Intervalos
intradiários (como `15m`) são o mais perto de "tempo real" que o free-tier oferece,
mas exigem janelas curtas (até ~60 dias).

In [5]:
# Um mês de candles diários
mensal = petr.history(period="1mo", interval="1d")
print(mensal.tail())

# Candles de 15 minutos dos últimos 5 dias (intradiário)
intraday = petr.history(period="5d", interval="15m")
print(intraday.tail())

                                Open       High        Low      Close  \
Date                                                                    
2026-10-30 00:00:00-03:00  52.450001  53.660000  52.360001  52.709999   
2026-11-03 00:00:00-03:00  51.200001  51.209999  49.150002  50.709999   
2026-11-04 00:00:00-03:00  50.040001  50.310001  46.639999  47.119999   
2026-11-05 00:00:00-03:00  47.900002  48.040001  45.950001  45.950001   
2026-11-06 00:00:00-03:00  46.000000  46.110001  44.959999  45.360001   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-10-30 00:00:00-03:00  61743200        0.0           0.0  
2026-11-03 00:00:00-03:00  93496300        0.0           0.0  
2026-11-04 00:00:00-03:00  94219800        0.0           0.0  
2026-11-05 00:00:00-03:00  53109700        0.0           0.0  
2026-11-06 00:00:00-03:00  47069700        0.0           0.0  
                                Open       High

## 7. Vários ativos de uma vez com `yf.download()`

Para comparar ativos, `yf.download()` baixa todos juntos. Com `group_by="ticker"`
as colunas ficam organizadas por papel.

In [6]:
dados = yf.download(
    ["PETR4.SA", "VALE3.SA", "ITUB4.SA"],
    period="5d",
    group_by="ticker",
    auto_adjust=True,
    progress=False,
)
dados

Ticker       PETR4.SA                                              VALE3.SA  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2026-10-30  52.450001  53.660000  52.360001  52.709999  61743200  59.439999   
2026-11-03  51.200001  51.209999  49.150002  50.709999  93496300  61.509998   
2026-11-04  50.040001  50.310001  46.639999  47.119999  94219800  61.599998   
2026-11-05  47.900002  48.040001  45.950001  45.950001  53109700  62.950001   
2026-11-06  46.000000  46.110001  44.959999  45.360001  47069700  62.599998   

Ticker                                                  ITUB4.SA             \
Price            High        Low      Close    Volume       Open       High   
Date                                                                          
2026-10-30  62.980000  59.419998  62.020000  34856400  39.181580  39.321514   
2026-11-03  62.700001  60.209999  60.250000  21064100  38.470001  38.770000   
2026-11-04  62.889999  61.230000  62.349998  27244600  39.299999  39.900002   
2026-11-05  63.160000  62.150002  62.669998  23599400  39.709999  40.169998   
2026-11-06  64.279999  62.020000  64.040001  19016100  39.980000  41.009998   

Ticker                                      
Price             Low      Close    Volume  
Date                                        
2026-10-30  38.671818  38.671818  51157200  
2026-11-03  38.110001  38.490002  43485400  
2026-11-04  38.849998  39.709999  47053600  
2026-11-05  39.330002  39.970001  32694800  
2026-11-06  39.869999  40.529999  27992000

## 8. Calculando a variação diária (a lógica usada no projeto)

Esta é exatamente a conta que a tool do assistente faz: pega o último fechamento,
o fechamento anterior, e calcula a variação percentual.

In [7]:
def variacao_diaria(ticker):
    hist = yf.Ticker(ticker).history(period="5d")
    if hist.empty:
        return f"{ticker}: sem dados"
    fech = hist["Close"].iloc[-1]
    ant = hist["Close"].iloc[-2] if len(hist) >= 2 else fech
    var = (fech - ant) / ant * 100
    return f"{ticker}: R$ {fech:.2f} ({var:+.2f}%)"

for t in ["PETR4.SA", "VALE3.SA", "ITUB4.SA"]:
    print(variacao_diaria(t))

PETR4.SA: R$ 45.36 (-1.28%)
VALE3.SA: R$ 64.04 (+2.19%)
ITUB4.SA: R$ 40.53 (+1.40%)


## 9. O índice Ibovespa (`^BVSP`)

O índice em si não leva `.SA`. Útil para dar contexto geral de mercado no relatório.

In [8]:
yf.Ticker("^BVSP").history(period="5d")

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-10-30 00:00:00-03:00,169185.000000,169185.000000,167008.000000,167922.000000,16364700,0.0,0.0
2026-11-03 00:00:00-03:00,167917.000000,168006.000000,164447.000000,165529.000000,15846300,0.0,0.0
2026-11-04 00:00:00-03:00,165548.000000,167986.000000,165541.000000,167564.000000,14959800,0.0,0.0
2026-11-05 00:00:00-03:00,167565.000000,168994.000000,166190.000000,166192.000000,12398400,0.0,0.0
2026-11-06 00:00:00-03:00,166191.453125,169073.046875,165909.734375,168498.453125,0,0.0,0.0


## 10. Armadilhas comuns

- **DataFrame vazio**: ticker errado, ativo sem pregão no período, ou instabilidade do
  Yahoo. Sempre cheque `hist.empty` antes de acessar `.iloc[-1]`.
- **`.info` vazio ou lento**: prefira `.history()` / `.fast_info` para preço.
- **Rate limiting**: muitas chamadas seguidas podem ser bloqueadas temporariamente.
  Busque vários tickers de uma vez com `yf.download()`.
- **Fuso horário**: o índice das datas vem com timezone; ao comparar com `datetime`
  "ingênuo" (sem tz) pode dar erro.
- **`auto_adjust`**: nas versões recentes vem `True` por padrão, então `Close` já é o
  preço ajustado por proventos. Para variação diária isso não atrapalha.